# 07a - Training-stability runs for the selected LoRA configurations

The hyperparameter search in 06 selected one configuration per pipeline on validation macro13:

- **direct**: `r=16, alpha=32, lr=2e-4, epochs=2` (val macro13 0.845)
- **structure**: `r=32, alpha=32, lr=2e-4, epochs=2` (val macro13 0.875)

Those are single training runs (seed 42). This notebook retrains each selected configuration with training seeds **123** and **456** and scores every run on the same 395 validation reports with the frozen rule. It measures how much the validation score moves when only the training seed changes.


In [ ]:
!pip install -q unsloth==2026.9.9
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

import unsloth, torch, transformers, trl, peft, bitsandbytes
print("unsloth      ", unsloth.__version__)
print("transformers ", transformers.__version__)
print("torch        ", torch.__version__)
print("trl          ", trl.__version__)
print("GPU          ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Switch the Colab runtime to GPU (T4) before running this notebook."


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth       2026.9.9
transformers  5.5.0
torch         2.11.0+cu130
trl           0.24.0
GPU           Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, gc, inspect
from datetime import datetime, timezone
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/cxr_eval_data"
SEARCH = os.path.join(BASE, "lora_search")
STAB = os.path.join(BASE, "stability")
os.makedirs(STAB, exist_ok=True)

need = ["sft_direct.jsonl", "sft_structure.jsonl", "prompt_templates.json", "train_set.csv", "val_set.csv",
        os.path.join("lora_search", "selected_configs.json")]
missing = [n for n in need if not os.path.exists(os.path.join(BASE, n))]
print(f"Found {len(need) - len(missing)}/{len(need)} required files in {BASE}")
if missing:
    for n in missing:
        print("  MISSING", n)
    raise FileNotFoundError("Run 05 and 06 first, then rerun this cell.")

with open(os.path.join(SEARCH, "selected_configs.json"), encoding="utf-8") as f:
    SELECTED = json.load(f)
assert set(SELECTED) == {"direct", "structure"}, set(SELECTED)
for pipe in SELECTED:
    for k in ("r", "alpha", "lr", "epochs", "config"):
        assert k in SELECTED[pipe], (pipe, k)
print("Selected configurations:", {p: SELECTED[p]["config"] for p in SELECTED})

PC = ['No Finding','Enlarged Cardiomediastinum','Cardiomegaly','Lung Lesion','Lung Opacity','Edema',
      'Consolidation','Pneumonia','Atelectasis','Pneumothorax','Pleural Effusion','Pleural Other',
      'Fracture','Support Devices']
CLIN = [p for p in PC if p != 'No Finding']

PATH_COLS = {
    "lungs_and_airways.lung_opacity": "Lung Opacity", "lungs_and_airways.consolidation": "Consolidation",
    "lungs_and_airways.edema": "Edema", "lungs_and_airways.atelectasis": "Atelectasis",
    "lungs_and_airways.lung_lesion": "Lung Lesion", "lungs_and_airways.pneumonia": "Pneumonia",
    "pleura.pleural_effusion": "Pleural Effusion", "pleura.pneumothorax": "Pneumothorax",
    "pleura.pleural_other": "Pleural Other", "cardiovascular.cardiomegaly": "Cardiomegaly",
    "cardiovascular.enlarged_cardiomediastinum": "Enlarged Cardiomediastinum",
    "bones_and_chest_wall.fracture": "Fracture", "devices.support_devices": "Support Devices",
    "general.no_finding": "No Finding",
}
VMAP = {"present": "1", "absent": "0", "uncertain": "-1", "not_mentioned": "NaN", "nan": "NaN"}

def parse_direct(raw):
    parsed = {}
    for line in raw.strip().split("\n"):
        line = line.strip()
        if "=" in line:
            parts = line.split("=")
            if len(parts) == 2:
                name, value = parts[0].strip(), parts[1].strip()
                if name in PC and value in ("1", "0", "-1", "NaN", "nan"):
                    parsed[name] = value
    return [parsed.get(p, "NaN") for p in PC], len(parsed)

def parse_struct(raw):
    s, e = raw.find("{"), raw.rfind("}")
    if s == -1 or e <= s:
        return ["NaN"] * 14, 0
    try:
        data = json.loads(raw[s:e + 1])
    except Exception:
        return ["NaN"] * 14, 0
    out = {}
    for path, p in PATH_COLS.items():
        cur = data
        ok = True
        for k in path.split("."):
            if isinstance(cur, dict) and k in cur:
                cur = cur[k]
            else:
                ok = False
                break
        if ok and isinstance(cur, str) and cur.strip().lower() in VMAP:
            out[p] = VMAP[cur.strip().lower()]
    return [out.get(p, "NaN") for p in PC], len(out)

def parse_any(pipe, raw):
    return parse_direct(raw) if pipe == "direct" else parse_struct(raw)

with open(os.path.join(BASE, "prompt_templates.json"), encoding="utf-8") as f:
    TPL = json.load(f)
assert set(TPL) == {"direct", "structure"}

TRAIN = {}
for pipe in ("direct", "structure"):
    with open(os.path.join(BASE, f"sft_{pipe}.jsonl"), encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    assert len(rows) == 2768, (pipe, len(rows))
    TRAIN[pipe] = rows

train = pd.read_csv(os.path.join(BASE, "train_set.csv"))
val = pd.read_csv(os.path.join(BASE, "val_set.csv"))
assert len(train) == 2768 and len(val) == 395, (len(train), len(val))
assert set(train["id"]) & set(val["id"]) == set(), "train/val overlap"
for pipe in ("direct", "structure"):
    assert {r["id"] for r in TRAIN[pipe]} == set(train["id"]), pipe
print(f"train={len(train)}  val={len(val)}  SFT rows: direct={len(TRAIN['direct'])} structure={len(TRAIN['structure'])}")

def truth_arrays(df):
    out = {}
    for p in PC:
        t = df[p].map(lambda x: np.nan if pd.isna(x) else float(x)).to_numpy()
        out[p] = np.where(t == -1, np.nan, np.where(np.isnan(t), 0.0, t))
    return out

def macro_scores(values, truth):
    per = {}
    for j, p in enumerate(PC):
        t = truth[p]
        pr = np.array([1.0 if v[j] == "1" else 0.0 for v in values])
        tp = int(((t == 1) & (pr == 1)).sum())
        fp = int(((t == 0) & (pr == 1)).sum())
        fn = int(((t == 1) & (pr == 0)).sum())
        d = 2 * tp + fp + fn
        per[p] = 2 * tp / d if d else 0.0
    return {"macro13": float(np.mean([per[p] for p in CLIN])),
            "macro14": float(np.mean(list(per.values()))),
            "NoFinding": per["No Finding"]}

STAB_COLS = ["pipeline", "config", "r", "alpha", "lr", "epochs", "seed", "steps", "train_loss",
             "macro13", "macro14", "NoFinding", "valid14", "usable_any", "minutes",
             "adapter_dir", "results_csv", "unsloth", "transformers", "torch", "done_utc"]
STAB_MANIFEST = os.path.join(STAB, "stability_results.csv")

def load_manifest():
    if os.path.exists(STAB_MANIFEST):
        return pd.read_csv(STAB_MANIFEST)
    return pd.DataFrame(columns=STAB_COLS)

def append_manifest(rec):
    m = load_manifest()
    if len(m):
        m = m[~((m["pipeline"] == rec["pipeline"]) & (m["config"] == rec["config"]))]
        m = pd.concat([m, pd.DataFrame([rec])], ignore_index=True)
        m.to_csv(STAB_MANIFEST, index=False)
    else:
        pd.DataFrame([rec]).to_csv(STAB_MANIFEST, index=False)

def config_name(pipe, cfg, seed):
    return f"{pipe}_r{cfg['r']}_a{cfg['alpha']}_lr{cfg['lr']:g}_ep{cfg['epochs']}_s{seed}"

def is_done(pipe, cfg, seed):
    m = load_manifest()
    hit = m[(m["pipeline"] == pipe) & (m["config"] == config_name(pipe, cfg, seed))]
    return len(hit) > 0 and os.path.exists(str(hit.iloc[-1]["results_csv"]))

print("Scorer ready: uncertain truth excluded, blank -> absent, only present positive, all reports scored.")


Mounted at /content/drive
Found 6/6 required files in /content/drive/MyDrive/cxr_eval_data
Selected configurations: {'direct': 'direct_r16_a32_lr0.0002_ep2_s42', 'structure': 'structure_r32_a32_lr0.0002_ep2_s42'}
train=2768  val=395  SFT rows: direct=2768 structure=2768
Scorer ready: uncertain truth excluded, blank -> absent, only present positive, all reports scored.


## Training and inference functions

In [3]:
# Training and validation inference functions - 06 functions with the seed exposed.
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

MAX_SEQ = 2048
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
INSTR = "<|start_header_id|>user<|end_header_id|>\n\n"
RESP = "<|start_header_id|>assistant<|end_header_id|>\n\n"
MAX_NEW = {"direct": 200, "structure": 300}
GEN_BATCH = 8

def load_lora(cfg, seed):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="meta-llama/Llama-3.2-3B-Instruct",
        max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=cfg["r"], target_modules=TARGETS, lora_alpha=cfg["alpha"],
        lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
        random_state=seed,
    )
    return model, tokenizer

def build_text(tokenizer, row):
    head = tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]}],
        tokenize=False, add_generation_prompt=True)
    return head + row["completion"] + tokenizer.eos_token

def train_one(cfg, pipe, seed, tokenizer, model, outdir, smoke=False):
    if tokenizer.eos_token != "<|eot_id|>":
        raise RuntimeError(f"Unexpected EOS token: {tokenizer.eos_token!r}")
    texts = [build_text(tokenizer, r) for r in TRAIN[pipe]]
    ds = Dataset.from_list([{"text": t} for t in texts])
    args = SFTConfig(
        output_dir=outdir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=cfg["epochs"],
        max_steps=4 if smoke else -1,
        learning_rate=cfg["lr"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1 if smoke else 10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=seed,
        save_strategy="no",
        report_to="none",
        dataset_text_field="text",
        max_length=MAX_SEQ,
        packing=False,
    )
    common = dict(model=model, train_dataset=ds, args=args)
    if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
        trainer = SFTTrainer(processing_class=tokenizer, **common)
    else:
        trainer = SFTTrainer(tokenizer=tokenizer, **common)
    _masked = train_on_responses_only(trainer, instruction_part=INSTR, response_part=RESP)
    if _masked is not None:
        trainer = _masked

    sample = trainer.train_dataset[0]
    kept = [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]
    decoded = tokenizer.decode(kept)
    if not decoded.lstrip().startswith(("No Finding=", "{")):
        raise RuntimeError("Response-only mask check failed. Unmasked tokens start with: " + repr(decoded[:120]))
    if smoke:
        print("  mask check passed; first unmasked tokens:", repr(decoded.lstrip()[:40]))

    stats = trainer.train()
    return trainer, stats

@torch.no_grad()
def predict(pipe, model, tokenizer, df):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    texts = []
    for report in df["full_report"]:
        p = TPL[pipe]["prefix"] + str(report) + TPL[pipe]["suffix"]
        texts.append(tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True))
    lens = [len(tokenizer(t)["input_ids"]) for t in texts]
    print(f"  prompt tokens max={max(lens)} p95={int(np.percentile(lens, 95))}")
    if max(lens) > MAX_SEQ - 100:
        raise RuntimeError(f"Prompt too long for max_seq_length={MAX_SEQ}: max {max(lens)}")
    raws = []
    for i in range(0, len(texts), GEN_BATCH):
        enc = tokenizer(texts[i:i + GEN_BATCH], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SEQ).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW[pipe],
                             do_sample=False, repetition_penalty=1.1)
        gen = out[:, enc["input_ids"].shape[1]:]
        raws.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
        print(f"  generated {len(raws)}/{len(texts)}", end="\r")
    print()
    return raws

def run_seed(pipe, cfg, seed, smoke=False):
    name = config_name(pipe, cfg, seed)
    outdir = os.path.join(STAB, "_smoke" if smoke else pipe, name)
    os.makedirs(outdir, exist_ok=True)
    df = (val.head(8) if smoke else val).reset_index(drop=True)
    t0 = time.time()
    print(f"\n=== {name}{' (SMOKE)' if smoke else ''} ===")

    model, tokenizer = load_lora(cfg, seed)
    trainer, stats = train_one(cfg, pipe, seed, tokenizer, model, outdir, smoke=smoke)
    raws = predict(pipe, model, tokenizer, df)

    parsed = [parse_any(pipe, r) for r in raws]
    values = [x[0] for x in parsed]
    found = [x[1] for x in parsed]
    ms = macro_scores(values, truth_arrays(df))

    out_rows = []
    for i, raw in enumerate(raws):
        s = df.iloc[i]
        row = {"id": s["id"]}
        for p in PC:
            row[f"truth_{p}"] = s[p]
        for j, p in enumerate(PC):
            row[f"pred_{p}"] = values[i][j]
        row["raw"] = raw
        row["n_found"] = found[i]
        out_rows.append(row)
    res_csv = os.path.join(outdir, "val_predictions.csv")
    pd.DataFrame(out_rows).to_csv(res_csv, index=False)

    adapter_dir = os.path.join(outdir, "adapter")
    if not smoke:
        model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)

    minutes = (time.time() - t0) / 60
    import unsloth, transformers
    rec = {
        "pipeline": pipe, "config": name, "r": cfg["r"], "alpha": cfg["alpha"],
        "lr": cfg["lr"], "epochs": cfg["epochs"], "seed": seed,
        "steps": int(trainer.state.global_step),
        "train_loss": float(stats.training_loss),
        "macro13": ms["macro13"], "macro14": ms["macro14"], "NoFinding": ms["NoFinding"],
        "valid14": float(np.mean([f == 14 for f in found])),
        "usable_any": float(np.mean([f > 0 for f in found])),
        "minutes": minutes,
        "adapter_dir": None if smoke else adapter_dir,
        "results_csv": res_csv,
        "unsloth": unsloth.__version__,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "done_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    if not smoke:
        append_manifest(rec)

    print(f"  macro13={ms['macro13']:.3f}  macro14={ms['macro14']:.3f}  "
          f"NoFind={ms['NoFinding']:.3f}  valid14={rec['valid14']*100:.1f}%  "
          f"usable={rec['usable_any']*100:.1f}%  ({minutes:.1f} min)")
    if smoke:
        print("  raw sample:", raws[0][:200].replace("\n", " | "))
        print("  smoke run: adapter not saved, manifest not updated")

    del trainer, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return rec


In [ ]:
run_seed("direct", {k: SELECTED["direct"][k] for k in ("r", "alpha", "lr", "epochs")}, 123, smoke=True)
print("\nSeed smoke test finished. If no error appeared, run the stability cell.")



=== direct_r16_a32_lr0.0002_ep2_s123 (SMOKE) ===
==((====))==  Unsloth 2026.9.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2768 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


... (5 outputs left out)

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.197820
2,0.232439
3,0.121989
4,0.129354


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  prompt tokens max=389 p95=368
  generated 8/8
  macro13=0.051  macro14=0.048  NoFind=0.000  valid14=100.0%  usable=100.0%  (2.1 min)
  raw sample: No Finding=NaN | Enlarged Cardiomediastinum=NaN | Cardiomegaly=NaN | Lung Lesion=-1 | Lung Opacity=-1 | Edema=NaN | Consolidation=NaN | Pneumonia=NaN | Atelectasis=NaN | Pneumothorax=0 | Pleural Effusion=NaN | Pleural Othe
  smoke run: adapter not saved, manifest not updated

Seed smoke test finished. If no error appeared, run the stability cell.


## Stability runs (seeds 123 and 456)

In [ ]:
SEEDS = [123, 456]

for pipe in ["direct", "structure"]:
    cfg = {k: SELECTED[pipe][k] for k in ("r", "alpha", "lr", "epochs")}
    for seed in SEEDS:
        if is_done(pipe, cfg, seed):
            print(f"skip {config_name(pipe, cfg, seed)} - already finished")
            continue
        run_seed(pipe, cfg, seed)

m = load_manifest()
print("\nCurrent stability manifest:")
print(m[["pipeline", "config", "seed", "macro13", "macro14", "NoFinding", "valid14", "minutes"]].to_string(index=False))


skip direct_r16_a32_lr0.0002_ep2_s123 - already finished
skip direct_r16_a32_lr0.0002_ep2_s456 - already finished

=== structure_r32_a32_lr0.0002_ep2_s123 ===
==((====))==  Unsloth 2026.9.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2768 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


... (118 outputs left out)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  generated 395/395


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cxr_eval_data/stability/structure/structure_r32_a32_lr0.0002_ep2_s456/adapter/tokenizer_config.json.


  macro13=0.842  macro14=0.850  NoFind=0.953  valid14=100.0%  usable=100.0%  (92.9 min)

Current stability manifest:
 pipeline                              config  seed  macro13  macro14  NoFinding  valid14    minutes
   direct    direct_r16_a32_lr0.0002_ep2_s123   123 0.852223 0.859092   0.948387      1.0  82.460404
   direct    direct_r16_a32_lr0.0002_ep2_s456   456 0.826376 0.835067   0.948052      1.0  76.168833
structure structure_r32_a32_lr0.0002_ep2_s123   123 0.876566 0.881415   0.944444      1.0 100.112359
structure structure_r32_a32_lr0.0002_ep2_s456   456 0.842221 0.850132   0.952978      1.0  92.921325


## Stability summary

In [ ]:
m = load_manifest()
s = pd.read_csv(os.path.join(SEARCH, "search_results.csv"))
rows = []

for pipe in ("direct", "structure"):
    sel = SELECTED[pipe]
    r42 = s[(s["pipeline"] == pipe) & (s["config"] == sel["config"])]
    assert len(r42) == 1, (pipe, len(r42))
    r42 = r42.iloc[0]
    rows.append({"pipeline": pipe, "seed": 42, "selected": True,
                 "config": str(r42["config"]), "r": int(r42["r"]), "alpha": int(r42["alpha"]),
                 "lr": float(r42["lr"]), "epochs": int(r42["epochs"]),
                 "macro13": float(r42["macro13"]), "macro14": float(r42["macro14"]),
                 "NoFinding": float(r42["NoFinding"]), "valid14": float(r42["valid14"]),
                 "minutes": float(r42["minutes"])})
    for seed in (123, 456):
        hit = m[(m["pipeline"] == pipe) & (m["seed"] == seed)]
        if len(hit) != 1:
            print(f"WARNING: {pipe} seed {seed}: {len(hit)} rows in the stability manifest")
            continue
        h = hit.iloc[0]
        rows.append({"pipeline": pipe, "seed": seed, "selected": False,
                     "config": str(h["config"]), "r": int(h["r"]), "alpha": int(h["alpha"]),
                     "lr": float(h["lr"]), "epochs": int(h["epochs"]),
                     "macro13": float(h["macro13"]), "macro14": float(h["macro14"]),
                     "NoFinding": float(h["NoFinding"]), "valid14": float(h["valid14"]),
                     "minutes": float(h["minutes"])})

df = pd.DataFrame(rows).sort_values(["pipeline", "seed"]).reset_index(drop=True)
print(df[["pipeline", "seed", "selected", "config", "macro13", "macro14", "NoFinding", "valid14", "minutes"]].to_string(index=False))

print("\nSeed variability on validation (all 395 reports):")
for pipe in ("direct", "structure"):
    sub = df[df["pipeline"] == pipe]
    unsel = sub.loc[sub["seed"].isin([123, 456]), "macro13"].to_numpy(dtype=float)
    all3 = sub["macro13"].to_numpy(dtype=float)
    if len(unsel) == 2:
        print(f"  {pipe}: seed42={sub.loc[sub['seed']==42,'macro13'].iloc[0]:.3f}  "
              f"seeds123/456 mean={unsel.mean():.3f} sd={unsel.std(ddof=1):.3f} "
              f"range={unsel.min():.3f}-{unsel.max():.3f}  "
              f"all3 mean={all3.mean():.3f} sd={all3.std(ddof=1):.3f}")
    else:
        print(f"  {pipe}: incomplete ({len(unsel)}/2 unselected seeds)")

out_path = os.path.join(STAB, "stability_summary.csv")
df.to_csv(out_path, index=False)
print("\nWrote", out_path)
print("Seed 42 was selected on validation and is optimistic. Seeds 123/456 describe training stability, not test uncertainty.")


 pipeline  seed  selected                              config  macro13  macro14  NoFinding  valid14    minutes
   direct    42      True     direct_r16_a32_lr0.0002_ep2_s42 0.844913 0.853431   0.964169 1.000000  75.320054
   direct   123     False    direct_r16_a32_lr0.0002_ep2_s123 0.852223 0.859092   0.948387 1.000000  82.460404
   direct   456     False    direct_r16_a32_lr0.0002_ep2_s456 0.826376 0.835067   0.948052 1.000000  76.168833
structure    42      True  structure_r32_a32_lr0.0002_ep2_s42 0.875457 0.880804   0.950311 0.997468  80.953805
structure   123     False structure_r32_a32_lr0.0002_ep2_s123 0.876566 0.881415   0.944444 1.000000 100.112359
structure   456     False structure_r32_a32_lr0.0002_ep2_s456 0.842221 0.850132   0.952978 1.000000  92.921325

Seed variability on validation (all 395 reports):
  direct: seed42=0.845  seeds123/456 mean=0.839 sd=0.018 range=0.826-0.852  all3 mean=0.841 sd=0.013
  structure: seed42=0.875  seeds123/456 mean=0.859 sd=0.024 range=0.842